# Model development: stream-fraud-detector

Exploratory companion to `training/generate_data.py` + `training/train.py` (the actual
production training path). This notebook re-runs the same functions so nothing here
can silently drift from what the model server loads — no separate feature logic.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd
import shap
from sklearn.ensemble import IsolationForest
from sklearn.metrics import RocCurveDisplay, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier

from common.features import FEATURE_NAMES
from training.generate_data import generate
from training.train import build_feature_matrix

## 1. Generate synthetic transactions (same generator the producer uses)

In [ ]:
rows = generate(n_users=500, tx_per_user=80, fraud_rate=0.02, seed=42)
df = pd.DataFrame(rows)
print(df.shape, "fraud rate:", df["is_fraud"].mean())
df.head()

## 2. Replay chronologically per-user to build rolling features

In [ ]:
X, y = build_feature_matrix(rows)
features_df = pd.DataFrame(X, columns=FEATURE_NAMES)
features_df["is_fraud"] = y
features_df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
features_df.boxplot(column="amount", by="is_fraud", ax=axes[0])
features_df.boxplot(column="amount_ratio_avg10", by="is_fraud", ax=axes[1])
plt.suptitle("")
plt.tight_layout()

## 3. Train XGBoost + IsolationForest ensemble

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="auc", random_state=42,
)
xgb.fit(X_train, y_train)

iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
iso.fit(X_train)
iso_scaler = MinMaxScaler(clip=True).fit(-iso.score_samples(X_train).reshape(-1, 1))

xgb_proba = xgb.predict_proba(X_test)[:, 1]
iso_score = iso_scaler.transform(-iso.score_samples(X_test).reshape(-1, 1)).ravel()
ensemble = 0.7 * xgb_proba + 0.3 * iso_score

print("XGBoost-only AUC:", roc_auc_score(y_test, xgb_proba))
print("Ensemble AUC:", roc_auc_score(y_test, ensemble))
print(classification_report(y_test, ensemble > 0.7))

In [ ]:
RocCurveDisplay.from_predictions(y_test, ensemble, name="ensemble")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.title("Ensemble ROC")

## 4. Feature importance

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=FEATURE_NAMES).sort_values()
importances.plot.barh(figsize=(6, 4), title="XGBoost feature importance")

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test[:500])
shap.summary_plot(shap_values, X_test[:500], feature_names=FEATURE_NAMES)

## 5. Save artifact

For the real pipeline, run `python -m training.generate_data && python -m training.train`
(or `docker compose run --rm trainer`) instead of saving from here — that's the
path the scorer's `models/model.joblib` actually comes from.